In [8]:
import os
from pathlib import Path
from datetime import date
import configparser
import pandas as pd
import geopandas as gpd
from sqlalchemy import text

from nvi_etl import working_dir, make_engine_for, setup_logging
from nvi_etl.utilities import fix_parcel_id
from nvi_etl.geo_reference import pull_zones, pull_council_districts, pin_location
from nvi_etl.reshape import elongate
from nvi_etl.destinations import SURVEY_VALUES_TABLE, CONTEXT_VALUES_TABLE

In [18]:
WORKING_DIR = Path(os.getcwd())

In [ ]:
TABLE_MAP = {
    "parcel_table": "raw.detodp_assessors_20260131", # this needs to have a geom, parcel_id, num_buildings, zoning_district column
    "parcel_det_table":"raw.detodp_assessor_20260131_det",
    "building_file_table": "raw.building_file_20230313_2",
    "blight_table":"raw.detodp_blight_violations_20260131",
    "mcm_table":"raw.survey_mcm_2014",
    "prop_conditions_table":"msc.nvi_prop_conditions_2025", 
    "valassis_1": "raw.valassis_vnefplus_mi_2025_qrt4_det", 
    "valassis_2": "raw.valassis_vnefplus_mi_2025_qrt3_det",
    "valassis_3": "raw.valassis_vnefplus_mi_20250501_det",
    "valassis_4": "raw.valassis_vnefplus_mi_20250122_det",
    "valassis_5": "raw.valassis_vnefplus_mi_20241017_det",
    "valassis_6": "raw.valassis_vnefplus_mi_20240711_det",
    "valassis_7": "raw.valassis_vnefplus_mi_20240411_det",
    "valassis_8": "raw.valassis_vnefplus_mi_20240116_det",
    "building_permits_table": "raw.detodp_building_permits_20260202",
}

db_engine = make_engine_for("ipds")

config = configparser.ConfigParser()
config.read(WORKING_DIR / "conf" / ".conf")

raw = "SELECT * FROM {parcel_table};".format(**TABLE_MAP)
sql = text(raw)

parcels = gpd.read_postgis(
    sql,
    db_engine,
    geom_col='geom',
    crs="EPSG:4326"
).to_crs(2898)

# nvi_zones = pull_zones(GEOM_DATE.year)
# council_districts = pull_council_districts(GEOM_DATE.year)



In [27]:
raw

'SELECT * FROM raw.detodp_assessors_20260131'

In [22]:
foreclosures = pd.read_csv(config["source_files"]["foreclosures_file"])

In [25]:
foreclosures["city_name"]

0                  DETROIT
1                  DETROIT
2                  DETROIT
3                  DETROIT
4                  DETROIT
               ...        
4411    VAN BUREN TOWNSHIP
4412    VAN BUREN TOWNSHIP
4413    VAN BUREN TOWNSHIP
4414         VAN BUREN TWP
4415    VAN BUREN TOWNSHIP
Name: city_name, Length: 4416, dtype: object

In [ ]:
tax_foreclosures = (
    pd.read_csv(config["source_files"]["foreclosures_file"])
    .query("city_name == 'DETROIT'")
    .dropna(subset=["parcel_id"])
    .astype({"parcel_id": "str"})
    .rename(columns={"parcel_id": "__parcel_id"})
    .assign(parcel_id=lambda df: df["__parcel_id"].apply(fix_parcel_id))
)

stamped = (
    parcels
    .merge(tax_foreclosures, on="parcel_id", how="left")
    .assign(not_in_foreclosure=lambda df: df["parcel_id"].isna())
    # .sjoin(council_districts[["district_number", "geometry"]], predicate="within", how="left")
    # .drop("index_right", axis=1)
    # .sjoin(nvi_zones[["zone_id", "geometry"]], predicate="within", how="left")
    # .drop("index_right", axis=1)
) 